# Nifty50 Weekly Expiry-To-Expiry Move Table (Skip First Candle)

For every consecutive pair of weekly expiries, this notebook measures how far Nifty50 traveled from the previous expiry's close before the next expiry — the max upside % (from the high) and max downside % (from the low) — plus the net close-to-close move at expiry.

**Variant note:** the measurement window skips the first trading day after each expiry and starts from the 2nd candle after expiry through the upcoming expiry (inclusive). The reference `base_close` is still the previous expiry's close.

Source data:
- `data/raw/option_selling/nifty/nifty_weekly_expiry_dates.csv` — scheduled and actual weekly expiry dates
- `data/raw/option_selling/nifty/nifty50_2020-01-01_2026-09-09.json` — daily OHLC candles

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

def find_project_root(start: Path | None = None) -> Path:
    """Walk up from the notebook's location until the folder holding data/raw is found."""
    path = (start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "option_selling" / "nifty"

EXPIRY_CSV = RAW_DIR / "nifty_weekly_expiry_dates.csv"
CANDLES_JSON = RAW_DIR / "nifty50_2020-01-01_2026-09-09.json"

pd.options.display.float_format = "{:,.2f}".format

## Load Raw Data

In [2]:
expiry_df = pd.read_csv(EXPIRY_CSV, parse_dates=["Actual_Expiry_Date"])
expiry_df = expiry_df.sort_values("Actual_Expiry_Date").reset_index(drop=True)
expiry_dates = expiry_df["Actual_Expiry_Date"].dt.normalize()

with open(CANDLES_JSON) as f:
    raw = json.load(f)

candles = raw["data"]["candles"]
price_df = pd.DataFrame(candles, columns=["timestamp", "open", "high", "low", "close", "volume", "oi"])
price_df["timestamp"] = pd.to_datetime(price_df["timestamp"]).dt.tz_localize(None).dt.normalize()
price_df = price_df.drop_duplicates("timestamp").set_index("timestamp").sort_index()
price_df = price_df[["open", "high", "low", "close"]].apply(pd.to_numeric)

# Restrict expiries to the range actually covered by the candle data
expiry_dates = expiry_dates[(expiry_dates >= price_df.index.min()) & (expiry_dates <= price_df.index.max())]

print(f"Loaded {len(price_df):,} daily candles from {price_df.index.min().date()} to {price_df.index.max().date()}")
print(f"{len(expiry_dates)} weekly expiries in range")
display(price_df.head())

Loaded 1,662 daily candles from 2020-01-01 to 2026-09-08
350 weekly expiries in range


,open,high,low,close
timestamp,,,,
2020-01-01,"12,202.15","12,222.20","12,165.30","12,182.50"
2020-01-02,"12,198.55","12,289.90","12,195.25","12,282.20"
2020-01-03,"12,261.10","12,265.60","12,191.35","12,226.65"
2020-01-06,"12,170.60","12,179.10","11,974.20","11,993.05"
2020-01-07,"12,079.10","12,152.15","12,005.35","12,052.95"


## Compute Max Up / Max Down Move Per Expiry-To-Expiry Week

For each window starting from the **2nd trading day after the previous expiry** through `current expiry` (inclusive) — i.e. the first candle right after expiry is skipped:
- **base_close** — Nifty50 close on the previous expiry
- **max_up_pct** — best case gain, `(highest high in window − base_close) / base_close`
- **max_down_pct** — worst case drop, `(lowest low in window − base_close) / base_close`
- **expiry_move_pct** — actual close-to-close move realized at the next expiry

In [3]:
rows = []
for prev_expiry, curr_expiry in zip(expiry_dates.iloc[:-1], expiry_dates.iloc[1:]):
    if prev_expiry not in price_df.index:
        continue

    full_window = price_df.loc[(price_df.index > prev_expiry) & (price_df.index <= curr_expiry)]
    # Skip the first trading day after expiry — start from the 2nd candle after expiry
    window = full_window.iloc[1:]
    if window.empty:
        continue

    base_close = price_df.loc[prev_expiry, "close"]
    max_up_date = window["high"].idxmax()
    max_down_date = window["low"].idxmin()
    expiry_close = window["close"].iloc[-1]

    rows.append(
        {
            "prev_expiry": prev_expiry,
            "curr_expiry": curr_expiry,
            "base_close": base_close,
            "max_up_pct": (window.loc[max_up_date, "high"] - base_close) / base_close * 100,
            "max_up_date": max_up_date,
            "max_down_pct": (window.loc[max_down_date, "low"] - base_close) / base_close * 100,
            "max_down_date": max_down_date,
            "expiry_close": expiry_close,
            "expiry_move_pct": (expiry_close - base_close) / base_close * 100,
            "trading_days": len(window),
        }
    )

moves = pd.DataFrame(rows)
moves["year"] = moves["curr_expiry"].dt.year
moves["range_pct"] = moves["max_up_pct"] - moves["max_down_pct"]

print(f"{len(moves)} expiry-to-expiry weeks computed")
display(moves.tail(10))

347 expiry-to-expiry weeks computed


,prev_expiry,curr_expiry,base_close,max_up_pct,max_up_date,max_down_pct,max_down_date,expiry_close,expiry_move_pct,trading_days,year,range_pct
337,2026-06-30,2026-07-07,"23,865.75",2.79,2026-07-07,0.81,2026-07-02,"24,398.70",2.23,4,2026,1.98
338,2026-07-07,2026-07-14,"24,398.70",-0.57,2026-07-13,-1.94,2026-07-09,"24,052.05",-1.42,4,2026,1.37
339,2026-07-14,2026-07-21,"24,052.05",1.31,2026-07-17,-0.01,2026-07-16,"24,187.70",0.56,4,2026,1.32
340,2026-07-21,2026-07-28,"24,187.70",-0.61,2026-07-28,-2.40,2026-07-24,"23,985.35",-0.84,4,2026,1.80
341,2026-07-28,2026-08-04,"23,985.35",3.29,2026-08-03,0.84,2026-07-30,"24,614.90",2.62,4,2026,2.45
342,2026-08-04,2026-08-11,"24,614.90",0.25,2026-08-06,-0.75,2026-08-11,"24,471.70",-0.58,4,2026,1.01
343,2026-08-11,2026-08-18,"24,471.70",-0.16,2026-08-13,-1.29,2026-08-18,"24,154.90",-1.29,4,2026,1.13
344,2026-08-18,2026-08-25,"24,154.90",0.74,2026-08-25,-0.16,2026-08-25,"24,334.55",0.74,4,2026,0.91
345,2026-08-25,2026-09-01,"24,334.55",-0.15,2026-08-27,-1.57,2026-09-01,"24,055.80",-1.15,4,2026,1.42
346,2026-09-01,2026-09-08,"24,055.80",-0.13,2026-09-03,-1.80,2026-09-08,"23,635.10",-1.75,4,2026,1.67


## Summary Stats

In [4]:
summary = pd.DataFrame(
    [
        ("Weeks analyzed", len(moves)),
        ("Avg max up %", f"{moves['max_up_pct'].mean():.2f}%"),
        ("Avg max down %", f"{moves['max_down_pct'].mean():.2f}%"),
        ("Biggest single-week rally", f"{moves['max_up_pct'].max():.2f}% (expiry {moves.loc[moves['max_up_pct'].idxmax(), 'curr_expiry'].date()})"),
        ("Biggest single-week crash", f"{moves['max_down_pct'].min():.2f}% (expiry {moves.loc[moves['max_down_pct'].idxmin(), 'curr_expiry'].date()})"),
        ("Avg expiry-to-expiry close move", f"{moves['expiry_move_pct'].mean():.2f}%"),
        ("Weeks closed higher", f"{(moves['expiry_move_pct'] > 0).sum()} / {len(moves)}"),
        ("Weeks closed lower", f"{(moves['expiry_move_pct'] < 0).sum()} / {len(moves)}"),
    ],
    columns=["Metric", "Value"],
)
display(summary)

,Metric,Value
0,Weeks analyzed,347
1,Avg max up %,1.33%
2,Avg max down %,-1.21%
3,Biggest single-week rally,10.64% (expiry 2020-04-09)
4,Biggest single-week crash,-18.33% (expiry 2020-03-19)
5,Avg expiry-to-expiry close move,0.20%
6,Weeks closed higher,190 / 347
7,Weeks closed lower,157 / 347


## Visual Table — Max Up % / Max Down % Per Expiry Week

Rows are sorted most recent first. Cell shading encodes magnitude: deeper amber for a larger max upside, deeper blue for a larger max downside — the sign and value are always printed alongside the color, so direction never relies on color alone.

In [5]:
UP_COLOR = (231, 111, 81)     # amber — max upside
DOWN_COLOR = (33, 158, 188)   # blue — max downside
NEUTRAL = (247, 247, 245)


def shade(values: pd.Series, color: tuple[int, int, int]) -> list[str]:
    magnitude = values.abs()
    peak = magnitude.max() or 1.0
    intensity = (magnitude / peak).clip(0, 1)
    colors = []
    for t in intensity:
        rgb = [int(NEUTRAL[i] + (color[i] - NEUTRAL[i]) * t) for i in range(3)]
        colors.append(f"rgb({rgb[0]},{rgb[1]},{rgb[2]})")
    return colors


table_df = moves.sort_values("curr_expiry", ascending=False).reset_index(drop=True)

up_colors = shade(table_df["max_up_pct"], UP_COLOR)
down_colors = shade(table_df["max_down_pct"], DOWN_COLOR)
close_colors = [
    "rgb(231,111,81)" if v > 0 else ("rgb(33,158,188)" if v < 0 else "rgb(247,247,245)")
    for v in table_df["expiry_move_pct"]
]

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[100, 100, 90, 90, 110, 90, 110, 70],
            header=dict(
                values=[
                    "Prev Expiry",
                    "Expiry",
                    "Base Close",
                    "Max Up %",
                    "Max Up Date",
                    "Max Down %",
                    "Max Down Date",
                    "Expiry Move %",
                ],
                fill_color="#2b2d42",
                font=dict(color="white", size=12),
                align="center",
                height=32,
            ),
            cells=dict(
                values=[
                    table_df["prev_expiry"].dt.strftime("%Y-%m-%d"),
                    table_df["curr_expiry"].dt.strftime("%Y-%m-%d"),
                    table_df["base_close"].map("{:,.2f}".format),
                    table_df["max_up_pct"].map("+{:.2f}%".format),
                    table_df["max_up_date"].dt.strftime("%Y-%m-%d"),
                    table_df["max_down_pct"].map("{:.2f}%".format),
                    table_df["max_down_date"].dt.strftime("%Y-%m-%d"),
                    table_df["expiry_move_pct"].map("{:+.2f}%".format),
                ],
                fill_color=[
                    ["white"] * len(table_df),
                    ["white"] * len(table_df),
                    ["white"] * len(table_df),
                    up_colors,
                    ["white"] * len(table_df),
                    down_colors,
                    ["white"] * len(table_df),
                    close_colors,
                ],
                align="center",
                height=26,
                font=dict(size=11),
            ),
        )
    ]
)
fig.update_layout(
    title="Nifty50 — Max Up / Max Down % From Prior Weekly Expiry",
    height=900,
    margin=dict(t=50, b=10, l=10, r=10),
)
fig.show()

## Max Up / Max Down % Across All Weeks (Diverging Bar)

In [6]:
chrono = moves.sort_values("curr_expiry")

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=chrono["curr_expiry"],
        y=chrono["max_up_pct"],
        name="Max up %",
        marker_color="rgb(231,111,81)",
    )
)
fig.add_trace(
    go.Bar(
        x=chrono["curr_expiry"],
        y=chrono["max_down_pct"],
        name="Max down %",
        marker_color="rgb(33,158,188)",
    )
)
fig.update_layout(
    title="Max Up / Max Down % From Prior Weekly Expiry, By Expiry Date",
    barmode="relative",
    height=480,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig.update_yaxes(title_text="% move from prior expiry close", ticksuffix="%")
fig.add_hline(y=0, line_color="#888", line_width=1)
fig.show()

## Return Distribution — Expiry-To-Expiry Close Move %

Buckets every weekly `expiry_move_pct` (prior expiry close → next expiry close) into 1% return-range bins, ordered best to worst, with a count/probability breakdown per bucket — the same style as a broker's option-probability distribution table.

In [7]:
import matplotlib

BIN_WIDTH = 1.0
LOWER_TAIL, UPPER_TAIL = -8.0, 6.0  # returns beyond this fall into the catch-all tail buckets

edges = np.arange(LOWER_TAIL, UPPER_TAIL + BIN_WIDTH, BIN_WIDTH)
bin_edges = [-np.inf, *edges, np.inf]

labels = [f"< {LOWER_TAIL:.0f}%"]
for lo, hi in zip(edges[:-1], edges[1:]):
    labels.append(f"{lo:.0f}% to {hi:.0f}%")
labels.append(f"> {UPPER_TAIL:.0f}%")

bucket = pd.cut(moves["expiry_move_pct"], bins=bin_edges, labels=labels, right=False)
dist = bucket.value_counts().reindex(labels[::-1]).rename("periods").to_frame()  # best bucket first

total = dist["periods"].sum()
dist["pct_of_total"] = dist["periods"] / total * 100
dist["cum_from_top"] = dist["periods"].cumsum()
dist["prob_at_least"] = dist["cum_from_top"] / total * 100          # P(return >= bucket's lower bound)
dist["prob_at_most"] = 100 - dist["prob_at_least"] + dist["pct_of_total"]  # P(return <= bucket's upper bound)
dist = dist.drop(columns="cum_from_top")

cmap = matplotlib.colormaps["RdYlGn"]
row_colors = [
    "rgb({:.0f},{:.0f},{:.0f})".format(*[c * 255 for c in cmap(t)[:3]])
    for t in np.linspace(1.0, 0.0, len(dist))  # best (green) at top, worst (red) at bottom
]
font_colors = ["black"] * len(dist)

fig = go.Figure(
    data=[
        go.Table(
            columnwidth=[130, 110, 90, 130, 130],
            header=dict(
                values=["Return Range", "Number of Periods", "% of Total", "Probability (≥ lower bound)", "Probability (≤ upper bound)"],
                fill_color="#2b2d42",
                font=dict(color="white", size=12),
                align="center",
                height=34,
            ),
            cells=dict(
                values=[
                    dist.index,
                    dist["periods"],
                    dist["pct_of_total"].map("{:.1f}%".format),
                    dist["prob_at_least"].map("{:.1f}%".format),
                    dist["prob_at_most"].map("{:.1f}%".format),
                ],
                fill_color=[row_colors] * 5,
                font=dict(color=[font_colors] * 5, size=11),
                align="center",
                height=28,
            ),
        )
    ]
)
fig.update_layout(
    title=f"Weekly Expiry-To-Expiry Return Distribution ({total} periods)",
    height=560,
    margin=dict(t=50, b=10, l=10, r=10),
)
fig.show()

## 🎯 Expiry Outliers & Global Events Analysis

This section identifies expiries where the expiry-to-expiry return fell outside a specified probability interval (e.g. 80% boundary). Outliers represent extreme market stress or expansion events. Drag the slider to change the probability boundary coverage and the scatter plot, stats, and table below all update together.

In [8]:
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import Markdown
from plotly.subplots import make_subplots

RETURN_COL = "expiry_move_pct"
RETURN_LABEL = "Weekly Return %"

BG = "#12141c"
GRID = "#262a3a"
WITHIN_COLOR = "#6c7a89"
DOWN_COLOR = "#ef4d6b"
UP_COLOR = "#8bc34a"
TEXT_COLOR = "#e8e8e8"

outlier_base = moves.sort_values("curr_expiry").reset_index(drop=True)


def classify_outliers(coverage_pct: int):
    tail = (100 - coverage_pct) / 2
    lower_q, upper_q = tail, 100 - tail
    lower_bound = np.percentile(outlier_base[RETURN_COL], lower_q)
    upper_bound = np.percentile(outlier_base[RETURN_COL], upper_q)

    df = outlier_base.copy()
    df["status"] = np.select(
        [df[RETURN_COL] < lower_bound, df[RETURN_COL] > upper_bound],
        ["Downside Outlier", "Upside Outlier"],
        default="Within Boundary",
    )
    return df, lower_bound, upper_bound, lower_q, upper_q


def render_outliers(coverage_pct=80):
    df, lower_bound, upper_bound, lower_q, upper_q = classify_outliers(coverage_pct)

    total = len(df)
    n_down = int((df["status"] == "Downside Outlier").sum())
    n_up = int((df["status"] == "Upside Outlier").sum())
    n_out = n_down + n_up

    display(Markdown(
        f"**Current Return Column:** `{RETURN_LABEL}` &nbsp;|&nbsp; **Total Records:** {total} expiries  \n"
        f"Based on the selected **{coverage_pct}%** coverage, the lower boundary is the "
        f"**{lower_q:.1f}th percentile** and the upper boundary is the **{upper_q:.1f}th percentile**."
    ))

    # --- Outliers Distribution (Survivability Scatter Plot) ---
    display(Markdown(
        "### ✈️ Outliers Distribution (Survivability Scatter Plot)\n"
        "Dots inside the horizontal dashed lines are within-boundary expiries. Dots outside are "
        "extreme outliers (red/green) — inspired by Abraham Wald's aircraft survivability analysis, "
        "highlighting where the market was hit by extreme moves."
    ))

    color_map = {"Within Boundary": WITHIN_COLOR, "Downside Outlier": DOWN_COLOR, "Upside Outlier": UP_COLOR}

    fig = go.Figure()
    for status, color in color_map.items():
        sub = df[df["status"] == status]
        fig.add_trace(
            go.Scatter(
                x=sub["curr_expiry"],
                y=sub[RETURN_COL],
                mode="markers",
                name=status,
                marker=dict(
                    color=color,
                    size=8 if status == "Within Boundary" else 11,
                    line=dict(color="white", width=1) if status != "Within Boundary" else None,
                ),
            )
        )

    fig.add_hline(
        y=upper_bound, line_dash="dash", line_color="#c9c9c9",
        annotation_text=f"Upper Boundary ({upper_bound:.2f}%)", annotation_position="top left",
        annotation_font_color="#c9c9c9",
    )
    fig.add_hline(
        y=lower_bound, line_dash="dash", line_color=DOWN_COLOR,
        annotation_text=f"Lower Boundary ({lower_bound:.2f}%)", annotation_position="bottom left",
        annotation_font_color=DOWN_COLOR,
    )

    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        height=480,
        xaxis_title="Expiry End Date",
        yaxis_title="Return (%)",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(t=40, b=40, l=50, r=20),
    )
    fig.update_xaxes(gridcolor=GRID)
    fig.update_yaxes(gridcolor=GRID, zerolinecolor=GRID)
    fig.show()

    stat_fig = make_subplots(rows=1, cols=4, specs=[[{"type": "indicator"}] * 4])
    stats = [
        ("Total Expiries", total, TEXT_COLOR, ""),
        ("Total Outliers", n_out, TEXT_COLOR, f" ({n_out / total * 100:.1f}%)" if total else ""),
        ("Downside Outliers", n_down, DOWN_COLOR, ""),
        ("Upside Outliers", n_up, UP_COLOR, ""),
    ]
    for i, (label, value, color, suffix) in enumerate(stats, start=1):
        stat_fig.add_trace(
            go.Indicator(
                mode="number",
                value=value,
                title={"text": label, "font": {"size": 14, "color": TEXT_COLOR}},
                number={"font": {"size": 30, "color": color}, "suffix": suffix},
            ),
            row=1, col=i,
        )
    stat_fig.update_layout(height=140, paper_bgcolor=BG, margin=dict(t=10, b=10, l=10, r=10))
    stat_fig.show()

    # --- Outlier Expiries Table ---
    display(Markdown(
        "### 🗂️ Outlier Expiries Table\n"
        "All weekly expiries that fell outside the specified boundaries, sorted by absolute size."
    ))

    out_df = df[df["status"] != "Within Boundary"].copy()
    out_df["abs_return"] = out_df[RETURN_COL].abs()
    out_df = out_df.sort_values("abs_return", ascending=False).drop(columns="abs_return")

    n_rows = len(out_df)
    status_colors = [DOWN_COLOR if s == "Downside Outlier" else UP_COLOR for s in out_df["status"]]

    table_fig = go.Figure(
        data=[
            go.Table(
                columnwidth=[90, 90, 110, 110],
                header=dict(
                    values=["Start Date", "End Date", "Weekly Return %", "Status"],
                    fill_color="#2b2d42",
                    font=dict(color="white", size=12),
                    align="center",
                    height=32,
                ),
                cells=dict(
                    values=[
                        out_df["prev_expiry"].dt.strftime("%Y-%m-%d"),
                        out_df["curr_expiry"].dt.strftime("%Y-%m-%d"),
                        out_df[RETURN_COL].map("{:+.2f}".format),
                        out_df["status"],
                    ],
                    fill_color=BG,
                    font=dict(
                        color=[[TEXT_COLOR] * n_rows, [TEXT_COLOR] * n_rows, [TEXT_COLOR] * n_rows, status_colors],
                        size=11,
                    ),
                    align="center",
                    height=28,
                ),
            )
        ]
    )
    table_fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=BG,
        height=min(600, 60 + 28 * max(n_rows, 1)),
        margin=dict(t=10, b=10, l=10, r=10),
    )
    table_fig.show()


coverage_slider = widgets.IntSlider(
    value=80, min=50, max=99, step=1,
    description="Probability Boundary Coverage (%)",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)
interact(render_outliers, coverage_pct=coverage_slider)

interactive(children=(IntSlider(value=80, continuous_update=False, description='Probability Boundary Coverage …

<function __main__.render_outliers(coverage_pct=80)>